# 導言

本文件是本研究**所有績效指標與統計檢定的單一權威定義來源**——
儀表板、`comparison.ipynb`、論文第四章引用的每一個數字，其定義都在這裡。
凡本文件與他處敘述不一致者，以本文件為準。

每項檢定一律以五欄說明：**公式 │ 虛無假設 │ 為何選它 │ 判讀門檻 │ 限制**。

**回測共同設定**：S&P 500 歷史成分股（Tiingo，2000–2025，動態成分股名冊避免存活者偏差）；
形成期 252 交易日、交易期 126 交易日、滾動步長 21 日；初始資本 \$10,000；
單邊交易成本 **0.29%**（進場、出場各按部位名目額扣一次，故每配對一完整往返 ≈ 0.58%），
依 Do 與 Faff（2012）對美股配對交易單邊成本約 30 bps 之估計。

# 報酬與資本口徑

配對交易為市場中性策略，資金**部分時間閒置**（無配對達到進場門檻時持有現金）。
因此「報酬」依分母（資本基準）不同而有多種口徑，各口徑對應不同文獻慣例。

## 累積與年化報酬

以每日投組損益 $\Delta_t$（各配對當日損益加總）建構權益曲線 $E_t = C_0 + \sum_{s\le t}\Delta_s$（$C_0=10{,}000$）。

- **累積報酬（Cum. Return）** $= \prod_{m}(1+r_m) - 1$，其中 $r_m$ 為月頻權益報酬。
- **年化報酬（Ann. Return）** $= (1+\text{Cum})^{12/n_m} - 1$，$n_m$ 為月數（幾何年化）。
- **Final Equity** $= C_0 + \sum_t \Delta_t$（期末權益）。

## 資本效率口徑（文獻慣例）

| 欄位 | 定義 | 文獻依據 |
|---|---|---|
| **RCC**（Return on Committed Capital） | 總損益 ÷ 承諾資本（$C_0$） | Gatev 等人（2006）保守口徑 |
| **REC**（Return on Employed Capital） | 總損益 ÷ 實際動用資本 | Gatev 等人（2006）fully-invested 口徑 |
| **Avg. Utilization** | 日均持倉配對數 ÷ 最大槽位（Top N × 重疊期數） | 資金利用率 |
| **Ann. Ret (Employed)** | 總損益 ÷（日均動用資金 × 年數） | GGR fully-invested 年化 |
| **Excess vs RF** | 承諾資本算術年化 − $r_f\times$利用率 | 閒置現金計無風險利率後的超額 |

> **判讀基準**：Gatev 等人（2006）於 1962–2002 樣本回報 committed 口徑約 11%／年；
> Do 與 Faff（2010）指出 2002 年後扣費淨值趨近於 0。本研究以 committed 口徑為主要基準，
> 並同列 employed 與 rf 超額口徑，忠實反映 post-2002 的報酬衰減。
> `RF_ANNUAL = 0.02`（config），約當 2000–2025 美國 3 個月期國庫券平均。


# 風險調整後指標

::: {.callout-important}

### 報酬口徑：一律複利（除以**前一日權益**）

$$r_t = \frac{\Delta_t}{E_{t-1}}, \qquad E_t = C_0 + \sum_{s \le t}\Delta_s$$

分母是**前一日權益**而非固定的初始資本 $C_0$。這是引擎實際的行為——
`portfolio_manager.allocate_capital` 以 `current_equity / max_pairs` 決定部位規模，
承擔風險的資本本來就隨權益走。

此口徑由 `strategies/returns.py` 單一提供，全部下游分析共用；
另有兩項規則一併由該模組負責：

- **生命期**：序列的起訖由該策略自己的首末交易日界定，**上線前不補零**
  （晚上線的策略族若被補零，$|$Sharpe$|$ 會被稀釋約 $\sqrt{\text{生命期}/\text{全期}}$）
- **跨策略對齊**：預設取生命期**交集**；重疊不足時拋錯，不靜默補零

:::

| 指標 | 公式 | 為何選它 | 判讀門檻 | 限制 |
|---|---|---|---|---|
| **Sharpe** | $\sqrt{252}\,\bar r / \sigma_r$ | 風險調整報酬的通用尺度；**尺度不變**，故不受資金規模影響 | >1 佳、>0.5 具實務意義、<0 劣 | 以全部波動為風險，對稱看待上下行；閒置現金日會稀釋 |
| **Sortino** | $\sqrt{252}\,\bar r / \sigma_r^{-}$，其中 $\sigma_r^{-} = \sqrt{\frac{1}{n}\sum_t \min(r_t, 0)^2}$（下行標準差） | 只以**下行**波動為風險——上漲的波動不該被罰 | 通常高於同策略 Sharpe；>1 佳 | 下行樣本較少故估計較不穩；跨策略比較需同樣本長度 |
| **Calmar** | 年化報酬 / \|MDD\| | 以「最壞情況」而非平均波動衡量風險，貼近實務可承受度 | >0.5 佳 | 由**單一**極端事件決定，樣本外極不穩定 |
| **Max Drawdown（MDD）** | $\min_t \dfrac{E_t - \max_{s\le t}E_s}{\max_{s\le t}E_s}$ | 直接回答「最多虧過多少」 | 越接近 0 越好（負值） | 與樣本長度正相關，長樣本天然更差 |
| **Information Ratio** | $\sqrt{252}\,\bar d / \sigma_d$，$d_t$ 為對基準的逐日報酬差 | 衡量**相對**基準的增益效率；同樣尺度不變 | >0.5 具意義 | 需與基準同期間、同配對底才可解讀 |

**Sharpe（Active Days Only）**：儀表板提供切換，僅計入「有持倉日」的報酬，
排除閒置現金日對波動的稀釋，與 buy-and-hold 的可比性較低但更反映策略活躍期的品質。

> Sharpe 判讀沿用 Sharpe（1994）；Sortino 依 Sortino 與 Price（1994）。
> 配對交易文獻（如 Do 與 Faff, 2010）於扣費後多落於 0–0.5 區間，
> 故本研究以 **>0.5 為具實務意義**、**顯著 > 基準**為主要目標。

# 交易統計

以「一筆完整交易」（進場至平倉）為單位，`Trade_PnL` 非零者計入。

| 欄位 | 定義 |
|---|---|
| **Win Rate** | 獲利交易數 ÷ 總交易數（≥50% 綠、<50% 紅） |
| **Profit Factor** | 總獲利 ÷ \|總虧損\|（>1 獲利、=1 損益兩平） |
| **Avg Hold (days)** | 平均持倉天數 |
| **Total Trades / Entries / Exits** | 交易筆數、進場次數、正常出場次數 |
| **Forced Closes** | 期末強制平倉次數（交易期結束仍持倉） |
| **Stop Losses** | 觸發停損次數 |
| **Gross Profit / Gross Loss** | 總獲利金額 / 總虧損金額 |

> **注意**：高 Win Rate 未必等於高獲利。若「贏小輸大」（平均獲利 < 平均虧損），
> 即使勝率 > 50%，Profit Factor 仍可能 < 1。本研究多數誠實策略即呈此結構，
> 這是配對交易在扣費後報酬趨零的微觀成因。


# 形成期篩選的三道檢定

篩選層對每組候選配對的價差（spread）依序施加，**任一未過即淘汰**。

| 檢定 | 公式 | 虛無假設 | 為何選它 | 判讀門檻 | 限制 |
|---|---|---|---|---|---|
| **ADF 共整合** | 對殘差 $\epsilon_t$ 估 $\Delta\epsilon_t = \lambda\epsilon_{t-1} + \dots$，檢定 $\lambda = 0$ | $H_0$：殘差**含單根**（不共整合） | Engle 與 Granger（1987）兩步驟法的第二步；價差若非定態，「會回歸」的前提就不成立 | $p < 0.05$（本研究基準；敏感度另試 0.01/0.10） | 對短樣本檢定力低；避險比率 $\beta$ 是估出來的，未計入其估計誤差 |
| **OU 半衰期** | $HL = -\ln 2 / \lambda$，$\lambda$ 取自上式的 AR(1) 係數 | —（非檢定，為篩選條件） | 共整合只說「終會回歸」，沒說**多快**；回歸太慢的配對在 126 日交易期內無法變現 | $1 \le HL \le 42$ 日（上限 = 交易期 126 日的 $1/3$） | AR(1) 為 OU 過程的離散近似；$HL$ 估計對樣本區間敏感 |
| **Hurst 指數** | $H$ 取自 R/S 統計量對尺度的對數斜率 | —（非檢定，為篩選條件） | 與 ADF 互補：ADF 看定態與否，$H$ 看**趨勢 vs 均值回歸傾向** | $H < 0.5$（$0.5$ = 隨機漫步、$>0.5$ = 趨勢） | 估計法眾多、結果不一致；短序列偏誤大 |

> 依據：Engle 與 Granger（1987）；OU 半衰期與 Hurst 的門檻設計依 Krauss 等人（2016）。

::: {.callout-warning}

### 排序與篩選的施行次序依後端而異

- **SSD 後端**（`ssd_rolling.py`）：先按 SSD 排序 → 取前 $\max(200,\ \text{Top}N\times15)$ 為候選
  → **依距離順序**逐一檢定 → 湊滿 $\text{Top}N\times5$ 即停 → 再排序取前 $N$
- **DTW／SSD-DTW-PCA 後端**（`DTW_Cointegration_Paper.py`）：群內**全配對**逐一檢定
  （無候選上限）→ 才排序取前 $N$

兩者的**共同**後果是：候選池不足時，系統會被迫接受距離更遠的配對。
但 SSD 路徑另有候選上限與提前中止，兩者不可互相引用其細節。

:::

# 統計檢定基準（T 檢定）

儀表板的 `T-Stat / p-val / NW T-Stat / NW p-val` 用以檢定**某策略是否顯著優於基準策略**，
而非僅比較單點績效（避免網格選擇偏差）。

## 虛無假設與統計量

對「策略 A」與「基準策略 B」在**共同月份**上計算逐月報酬差 $d_m = r^A_m - r^B_m$，檢定：

$$H_0:\ \mathbb{E}[d_m] = 0 \quad\text{vs}\quad H_1:\ \mathbb{E}[d_m] \ne 0$$

- **一般 t 檢定**：$t = \bar d / (s_d/\sqrt{n})$，$s_d$ 為 $d_m$ 樣本標準差。
- **Newey-West 修正 t 值**：以 Newey 與 West（1987）的 HAC 標準誤修正報酬的
  序列自相關與異質變異（落後階數預設 3），統計量更保守、更穩健。

## 判讀門檻

| p 值 | 標示 | 意義 |
|---|---|---|
| $p < 0.05$ | 綠色粗體 | 5% 水準顯著優於／異於基準 |
| $0.05 \le p < 0.10$ | 橙色 | 10% 水準邊際顯著 |
| $p \ge 0.10$ | 無標示 | 無法拒絕「與基準無差異」 |

> **基準策略的選定**：命題 1（形成法）以 **SSD Rolling**（距離／共整合家族基準）為對照；
> 命題 2（交易法）以**同一組配對的 Z-Score 回歸基準**為對照（DL-THR／距離策略借用相同形成期
> 配對，構成單變因對照）。因此正的且顯著的 t 值代表「該創新相對基準的淨增量貢獻」。
> 逐月配對檢定與 Newey-West 修正為資產定價實證的標準做法（Newey & West, 1987）。

::: {.callout-note}

### 儀表板與論文正文的檢定不同

本頁描述的是**儀表板**的逐月 t 檢定欄位。論文第四章的主檢定另有其設計：
以**逐日**報酬差搭配**循環 block bootstrap**（$L$=126，10,000 次重抽），
同時報告 $p$ 值與 95% 信賴區間，並一律以全網格等權組合為口徑。
兩者的抽樣頻率與推論方法皆不同，數字不可互相引用。

:::


# 論文正文的推論工具

上一節是**儀表板**的逐月 t 檢定。論文第四章的主檢定另成一套，定義如下。

## 主檢定：逐日報酬差 + 循環 block bootstrap

| 項目 | 內容 |
|---|---|
| **公式** | 對策略 A、B 在共同交易日上取 $d_t = r^A_t - r^B_t$；以區塊長度 $L = 126$ 對 $d_t$ **循環**重抽 10,000 次，得 $\bar d$ 的經驗分布 |
| **虛無假設** | $H_0:\ \mathbb{E}[d_t] = 0$ |
| **為何選它** | 逐日報酬有序列自相關與異質變異，i.i.d. 假設不成立。區塊重抽保留區塊內的相依結構；**循環**版本使每個觀測被抽中的機率相等，消除端點偏誤（Politis & Romano, 1992）。無母數，不需常態假設 |
| **判讀門檻** | 雙尾 $p < 0.05$；同時報告 95% 信賴區間 |
| **限制** | $L$ 的選擇是判斷而非事實（本研究取 $L$ = 交易期長度）；區塊過短會低估相依、過長則有效樣本數銳減 |

## 對照：Newey-West HAC

| 項目 | 內容 |
|---|---|
| **公式** | $t = \bar d\,/\,\mathrm{SE}_{\text{HAC}}$，$\mathrm{SE}_{\text{HAC}}$ 依 Newey 與 West（1987）以 Bartlett 核加權自相關項 |
| **虛無假設** | 同上 |
| **為何選它** | 資產定價實證的標準做法，與 block bootstrap 互為佐證；兩者結論一致時推論更可信 |
| **判讀門檻** | 落後階數取 $\lfloor 4(n/100)^{2/9}\rfloor$ 或固定 3（儀表板） |
| **限制** | 仍依賴漸近常態；小樣本下標準誤偏低、t 值偏大 |

## 多重檢定校正：Benjamini-Hochberg FDR

| 項目 | 內容 |
|---|---|
| **公式** | 將 $m$ 個 $p$ 值升冪排序，取最大的 $k$ 使 $p_{(k)} \le \frac{k}{m}q$；校正後 $p^{adj}_{(i)} = \min_{j \ge i}\left(\frac{m}{j}p_{(j)}\right)$ |
| **虛無假設** | 各對照的 $H_0$ 同時檢定 |
| **為何選它** | 命題 1 涉及 9 組對照。Bonferroni 控制 FWER 過於保守、在探索性研究中檢定力太低；BH 控制**偽發現率**，是此情境的慣例（Benjamini & Hochberg, 1995） |
| **判讀門檻** | $q = 0.05$；一律同時報告原始與校正後 $p$ |
| **限制** | 假設檢定間為獨立或正相關；本研究的對照共用同一批價格資料，相依性未經驗證 |

## 選擇偏誤校正：Deflated Sharpe Ratio

| 項目 | 內容 |
|---|---|
| **公式** | $\widehat{SR}_0 = \sqrt{V[\widehat{SR}]}\left[(1-\gamma)\Phi^{-1}\!\left(1-\tfrac{1}{N}\right) + \gamma\,\Phi^{-1}\!\left(1-\tfrac{1}{Ne}\right)\right]$（$\gamma$ = Euler–Mascheroni 常數）<br/>$\widehat{DSR} = \Phi\!\left(\dfrac{(\widehat{SR}-\widehat{SR}_0)\sqrt{T-1}}{\sqrt{1-\hat\gamma_3\widehat{SR}+\frac{\hat\gamma_4-1}{4}\widehat{SR}^2}}\right)$ |
| **虛無假設** | $H_0$：真實 Sharpe $\le 0$，觀察到的高 Sharpe 純由「在 $N$ 次試驗中取最大」造成 |
| **為何選它** | 在 $N$ 個配置中挑最佳者會系統性高估 Sharpe。DSR 依 $N$、試驗間 Sharpe 變異、報酬偏態與峰態、樣本長度，給出「真實 Sharpe > 0」的機率（Bailey & López de Prado, 2014） |
| **判讀門檻** | $DSR \ge 0.95$ 方可宣稱顯著為正 |
| **限制** | ⚠ **$N$ 是判斷而非事實**——試驗宇宙該涵蓋哪些封存實驗，沒有客觀答案。本研究因此以**全網格等權組合**為主要報告口徑（沒有東西可挑，選擇偏誤自源頭消失），DSR 僅列於附錄。$N$ 與試驗間變異在 `analysis/regime_cost_dsr_eval.py` 中**釘死為常數**並記錄清點日期，否則數字會隨資料庫累積而漂移、論文表無法重現 |

## 分群穩定度：調整蘭德指數（ARI）

| 項目 | 內容 |
|---|---|
| **公式** | $ARI = \dfrac{RI - \mathbb{E}[RI]}{\max(RI) - \mathbb{E}[RI]}$，$RI$ 為兩組分群在所有「兩兩配對是否同群」上的一致比例 |
| **虛無假設** | —（非檢定，為相似度量測） |
| **為何選它** | 群編號本身沒有意義（同一組分群換個編號仍是同一組），故比對「兩兩是否同群」而非標籤本身；ARI 已對隨機一致校正（Hubert & Arabie, 1985） |
| **判讀門檻** | $1$ = 完全相同、$0$ = 與隨機分派無異 |
| **限制** | 只量測**是否改變**，不量測改變**是否有益**；本研究無「靜態分群」對照組 |

## 成本穩健性：Break-even 往返成本

| 項目 | 內容 |
|---|---|
| **公式** | 進出場費 $= \text{friction} \times$ 名目額，且 $v_A + v_B = C_{\text{每配對}}$，故總費用 $F_0 = \text{friction}\times C_{\text{每配對}}\times(\text{進場}+\text{出場次數})$。單邊 $c^* = 0.29\% + \dfrac{\text{淨利}}{\sum\text{名目額}}$；往返 $= 2c^*$ |
| **虛無假設** | —（非檢定，為敏感度分析） |
| **為何選它** | Do 與 Faff（2012）的建議：報告「使淨利歸零的成本水準」，使結論不依賴單一成本假設 |
| **判讀門檻** | $c^*_{\text{往返}}$ 越高於現行假設 0.58% 越穩健 |
| **限制** | 精確可解但假設費用與名目額成固定比例，未計入市場衝擊隨部位規模的非線性 |

::: {.callout-important}

### 報告口徑：一律全網格等權組合

一切績效主張與統計檢定，皆以該策略 **15 個參數配置的等權組合**為口徑。
改報「網格最佳格」會內含 15 選 1 的選擇偏誤，必須再以 DSR 之類的程序扣回去——
而該程序的關鍵參數（試驗數 $N$）是判斷而非事實。
**等權組合沒有東西可挑，選擇偏誤自源頭消失。**

*例外*：regime 分層與 break-even 成本表以最佳格計算——
二者刻畫風險的**形狀**而非主張績效水準，且最佳格對其論證方向是保守的。

:::

# 機器學習與深度學習：特徵與訓練資料

本研究兩處使用學習方法：**形成期的非監督式分群**（命題 1）與**交易期的門檻選擇**（命題 2）。
以下完整說明各自的特徵、訓練資料與執行步驟。


## 形成期分群（非監督式）

**目的**：以資料驅動方式將行為相似的股票分群，縮小配對搜尋空間，取代靜態 GICS 產業分組。

### 特徵

1. **報酬 PCA 因子載荷**（HDBSCAN Cluster / Agglomerative）：對形成期日報酬矩陣（逐股
   標準化 → 等同對相關矩陣）做 PCA，取前 $k$ 個主成分（$k=5\sim15$），每檔股票的載荷
   向量（以 $\sqrt{\text{特徵值}}$ 加權）即其座標。此表徵反映股票在共同風險因子上的暴露
   （Avellaneda & Lee, 2010），是共整合關係的經濟基礎。
2. **公司基本面**（Agglomerative Fundamentals）：對數市值、盈餘殖利率（1/PE），
   以產業中位數插補缺失並 winsorize；Point-in-Time 對齊避免前視。
3. **GICS 產業 one-hot**：加權後併入特徵，軟性引導同產業靠近。

### 演算法與執行步驟

1. 建構特徵矩陣（上述 1–3，各區塊標準化後加權串接）。
2. **HDBSCAN**（Campello 等人, 2013）或 **Agglomerative**（Ward linkage）分群；
   HDBSCAN 以密度自動決定群數並辨識雜訊點，Agglomerative 以距離門檻（分位數）切群。
3. 群內全配對做共整合／距離篩選（ADF、OU 半衰期、Hurst、零穿越），依 SSD／DTW／
   PCA 融合分數排序取 Top N。

> **非監督式、無標籤**：分群不使用未來報酬，純以形成期資料分群，故無前視風險。


## 交易期門檻選擇（DL-THR 門檻選擇式 v4）

**目的**：每配對每期自適應選擇進出場門檻，取代固定的 Z-Score 門檻（entry_z=2, exit_z=0）。
設計依 Kim 與 Kim（2019）之門檻選擇框架，動作選單**包含靜態基準**，故策略空間為
Z-Score 基準之超集。

### 輸入特徵（12 維，形成期計算，標準化至約 $[-3,3]$）

以形成期 spread 的 Z 序列與兩檔股票的 log 價格計算：期末 z、$|$期末 z$|$、零穿越頻率、
OU 半衰期（對數）、近期 z 波動 regime、近期 z 趨勢、log 價格相關係數、報酬波動比、
對沖比率、spread 振幅、形成期 $|z|>2$ 佔比、$\max|z|$。這些特徵刻畫配對在交易期
**是否／如何**均值回歸，用以預測各門檻組合的報酬。

### 訓練資料與標籤（全資訊監督回歸）

- **動作選單（9）**：SKIP（不交易）＋ 8 組 $(\text{entry\_z},\text{exit\_z}) \in
  \{1.5,2.0,2.5,3.0\}\times\{0.0,0.5\}$。
- **標籤**：對每一個歷史配對期，**精確反事實回算**全部 9 個動作在該交易期的實際報酬
  （逐一以 Z-Score 狀態機模擬）。因報酬完全可算，此為**全資訊監督回歸**而非部分回饋的
  bandit——無探索問題、樣本效率最高。
- **網路**：MLP（12 → 隱藏層 → 9），輸出各動作的預期報酬，決策時取 argmax。

### Walk-forward 訓練步驟（無前視）

1. 依時間順序走過各滾動期。決策第 $k$ 期時，**僅**以「交易期已於第 $k$ 期開始前結束」
   的歷史配對期樣本訓練（滾動緩衝，重疊期自動排除）。
2. 訓練樣本不足 `thr_min_train_samples`（預設 200）時，**自動退回靜態基準動作 (2.0, 0.0)**，
   確保暖身期行為 $\equiv$ Z-Score 基準。
3. 以選定門檻交由標準 Z-Score 狀態機執行整個交易期，落地交易紀錄。

> **可證偽的比較框架**：因動作選單含基準，DL-THR 若學得當則 $\ge$ 基準、學不好至多退化為基準；
> 任何顯著正的 t 值即為「自適應門檻相對固定門檻」的淨貢獻。

::: {.callout-warning}

### 這不是強化學習，另有真正的部分回饋對照

因 9 個動作的報酬全部可反事實回算，本策略是**全資訊監督回歸**——
無探索、無價值迭代、無序列信用分配。`result.db` 的 `DRL` 欄位值為歷史代號。

**RL-THR**（`strategies/trading/rl_threshold_trading.py`）為其受控對照：
同一動作選單、同一 12 維狀態、同一網路與 walk-forward 切分，僅改兩處——
訓練標籤只保留**實際選中的那一個**動作（遮罩 MSE），決策改為 $\varepsilon$-greedy
（掃 $0.05$／$0.10$／$0.20\!\to\!0.02$ 三組）。用以量化**反事實標籤本身的價值**。

它是 contextual bandit，沒有 $\gamma$：每期一次決策，且 12 維狀態全由形成期視窗算出，
選哪個門檻不改變下一期狀態，沒有東西可以 bootstrap。

:::

# 參考文獻

*（APA 第 7 版）*

Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the US equities market. *Quantitative Finance, 10*(7), 761–782. https://doi.org/10.1080/14697680903124632

Bailey, D. H., & López de Prado, M. (2014). The deflated Sharpe ratio: Correcting for selection bias, backtest overfitting, and non-normality. *The Journal of Portfolio Management, 40*(5), 94–107. https://doi.org/10.3905/jpm.2014.40.5.094

Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: A practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B, 57*(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x

Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. In *Advances in Knowledge Discovery and Data Mining (PAKDD 2013)* (pp. 160–172). Springer. https://doi.org/10.1007/978-3-642-37456-2_14

Do, B., & Faff, R. (2010). Does simple pairs trading still work? *Financial Analysts Journal, 66*(4), 83–95. https://doi.org/10.2469/faj.v66.n4.1

Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research, 35*(2), 261–287. https://doi.org/10.1111/j.1475-6803.2012.01317.x

Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica, 55*(2), 251–276. https://doi.org/10.2307/1913236

Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative-value arbitrage rule. *The Review of Financial Studies, 19*(3), 797–827. https://doi.org/10.1093/rfs/hhj020

Hubert, L., & Arabie, P. (1985). Comparing partitions. *Journal of Classification, 2*(1), 193–218. https://doi.org/10.1007/BF01908075

Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity, 2019*, 3582516. https://doi.org/10.1155/2019/3582516

Krauss, C., Do, X. A., & Huck, N. (2016). Deep neural networks, gradient-boosted trees, random forests: Statistical arbitrage on the S&P 500. *European Journal of Operational Research, 259*(2), 689–702. https://doi.org/10.1016/j.ejor.2016.10.031

Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics, 17*(3), 1217–1241. https://doi.org/10.1214/aos/1176347265

Newey, W. K., & West, K. D. (1987). A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix. *Econometrica, 55*(3), 703–708. https://doi.org/10.2307/1913610

Politis, D. N., & Romano, J. P. (1992). A circular block-resampling procedure for stationary data. In R. LePage & L. Billard (Eds.), *Exploring the limits of bootstrap* (pp. 263–270). Wiley.

Sharpe, W. F. (1994). The Sharpe ratio. *The Journal of Portfolio Management, 21*(1), 49–58. https://doi.org/10.3905/jpm.1994.409501

Sortino, F. A., & Price, L. N. (1994). Performance measurement in a downside risk framework. *The Journal of Investing, 3*(3), 59–64. https://doi.org/10.3905/joi.3.3.59

Sutton, R. S., & Barto, A. G. (2018). *Reinforcement learning: An introduction* (2nd ed.). MIT Press.